<a href="https://colab.research.google.com/github/Faith-moses/DecodeLabs-Internship/blob/main/ecommerce_sales_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import Libraries, Load Dataset and Preview

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
orders= pd.read_excel("/content/drive/MyDrive/Dataset_for_Data_Analytics.xlsx")

In [4]:
#check dataset
orders.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [ ]:
#check rows and columns
orders.shape

(1200, 14)

In [ ]:
#check column names
orders.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

#Data Understanding

In [ ]:
#check data information
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [ ]:
#data statistical summary
orders.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [ ]:
#check missing values
orders.isnull().sum()

,0
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0


In [ ]:
#check duplicate rows
orders.duplicated().sum()

np.int64(0)

In [ ]:
orders['OrderID'].duplicated().sum()

np.int64(0)

In [ ]:
orders['TrackingNumber'].duplicated().sum()

np.int64(0)

#Data Validation and Anomalies Detection

In [ ]:
# Check if TotalPrice = Quantity × UnitPrice
orders['CalculatedTotal'] = orders['Quantity'] * orders['UnitPrice']
orders['PriceDiscrepancy'] = orders['TotalPrice'] - orders['CalculatedTotal']
discrepancies = orders[orders['PriceDiscrepancy'].abs() > 0.01]  # Allow tiny float errors

print(f"\n PRICE VALIDATION:")
print(f"   Orders where TotalPrice ≠ Quantity × UnitPrice: {len(discrepancies)}")
if len(discrepancies) > 0:
    print(f"   Sample discrepancies:")
    print(discrepancies[['OrderID', 'Quantity', 'UnitPrice', 'TotalPrice', 'CalculatedTotal', 'PriceDiscrepancy']].head())
    print(f"\n   Possible explanations:")
    print(f"   - Shipping costs included?")
    print(f"   - Coupon discounts applied?")
    print(f"   - Tax added?")

    # Check if discrepancy correlates with coupon
    orders['HasCoupon'] = orders['CouponCode'].notna()
    coupon_discrep = orders.groupby('HasCoupon')['PriceDiscrepancy'].mean()
    print(f"\n   Average discrepancy by coupon usage:")
    print(coupon_discrep)
else:
    print("   ✅ All prices calculate correctly")


 PRICE VALIDATION:
   Orders where TotalPrice ≠ Quantity × UnitPrice: 0
   ✅ All prices calculate correctly


In [ ]:
# Check date range and logic
print(f"\n DATE VALIDATION:")
print(f"   Date range: {orders['Date'].min()} to {orders['Date'].max()}")
print(f"   Total span: {(orders['Date'].max() - orders['Date'].min()).days} days")


 DATE VALIDATION:
   Date range: 2023-01-01 00:00:00 to 2025-06-30 00:00:00
   Total span: 911 days


In [ ]:
# Check for future dates (relative to today)
today = pd.Timestamp('2026-05-22')
future_orders = orders[orders['Date'] > today]
print(f"   Orders with future dates: {len(future_orders)}")

   Orders with future dates: 0


In [ ]:
#check cancelled orders with tracking
cancelled_with_tracking = orders[(orders['OrderStatus'] == 'Cancelled') & (orders['TrackingNumber'].notna())]
print(f" Cancelled orders WITH tracking numbers: {len(cancelled_with_tracking)}")

 Cancelled orders WITH tracking numbers: 250


In [ ]:
#check pending orders with tracking
pending_with_tracking = orders[(orders['OrderStatus'] == 'Pending') & (orders['TrackingNumber'].notna())]
print(f"   Pending orders WITH tracking numbers: {len(pending_with_tracking)}")

   Pending orders WITH tracking numbers: 237


In [ ]:
# Check if ItemsInCart < Quantity (illogical)
illogical_cart = orders[orders['ItemsInCart'] < orders['Quantity']]
print(f"\n   Orders where ItemsInCart < Quantity: {len(illogical_cart)}")


   Orders where ItemsInCart < Quantity: 0


#Data Cleaning and Feature Engineering

In [ ]:
#make a clean copy
orders_clean = orders.copy()

In [ ]:
#fill null couponcode with NONE
orders_clean['CouponCode'] = orders_clean['CouponCode'].fillna('NONE')

print("Filled missing coupon code with NONE")

Filled missing coupon code with NONE


In [ ]:
#create Year, Month, Quarter columns
orders_clean['Year'] = orders_clean['Date'].dt.year
orders_clean['Month'] = orders_clean['Date'].dt.month_name()
orders_clean['Quarter'] = orders_clean['Date'].dt.quarter

In [ ]:
#create unattended orders column
orders_clean['Unattendedorders'] = orders_clean['ItemsInCart'] - orders_clean['Quantity']
print(f"✅ Created UnattendedOrders (cart abandonment): mean = {orders_clean['Unattendedorders'].mean():.2f}")

✅ Created UnattendedOrders (cart abandonment): mean = 2.54


In [ ]:
customer_metrics = orders_clean.groupby('CustomerID').agg(
    TotalOrders=('OrderID', 'count'),
    TotalSpent=('TotalPrice', 'sum'),
    FirstOrder=('Date', 'min'),
    LastOrder=('Date', 'max'),
    AvgOrderValue=('TotalPrice', 'mean'),
    TotalQuantity=('Quantity', 'sum'),
    TotalCartItems=('ItemsInCart', 'sum'),
    TotalUnattended=('Unattendedorders', 'sum'),
    PreferredProduct=('Product', lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
).reset_index()

In [ ]:
customer_metrics['CustomerLifetimeDays'] = (customer_metrics['LastOrder'] - customer_metrics['FirstOrder']).dt.days
customer_metrics['IsRepeatCustomer'] = (customer_metrics['TotalOrders'] > 1).astype(int)
customer_metrics['CartAbandonmentRate'] = (customer_metrics['TotalUnattended'] / customer_metrics['TotalCartItems'] * 100).round(1)

In [ ]:
# Merge back
orders_clean = orders_clean.merge(
    customer_metrics[['CustomerID', 'TotalOrders', 'TotalSpent', 'AvgOrderValue',
                      'IsRepeatCustomer', 'CustomerLifetimeDays', 'CartAbandonmentRate']],
    on='CustomerID',
    how='left'
)

print(f"Customer metrics computed for {len(customer_metrics)} unique customers")
print(f"Repeat customers: {customer_metrics['IsRepeatCustomer'].sum()} ({customer_metrics['IsRepeatCustomer'].mean()*100:.1f}%)")
print(f"Average cart abandonment per customer: {customer_metrics['TotalUnattended'].mean():.2f} items")

Customer metrics computed for 1189 unique customers
Repeat customers: 11 (0.9%)
Average cart abandonment per customer: 2.56 items


In [ ]:
orders_clean['HasCoupon'] = (orders_clean['CouponCode'] != 'NONE').astype(int)
orders_clean['DayOfWeek'] = orders_clean['Date'].dt.day_name()
orders_clean['IsWeekend'] = orders_clean['Date'].dt.weekday.isin([5, 6]).astype(int)

In [ ]:
# Order status flags for analysis
orders_clean['IsCancelled'] = (orders_clean['OrderStatus'] == 'Cancelled').astype(int)
orders_clean['IsReturned'] = (orders_clean['OrderStatus'] == 'Returned').astype(int)
orders_clean['IsProblemOrder'] = orders_clean['OrderStatus'].isin(['Cancelled', 'Returned']).astype(int)

In [ ]:
orders_clean

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,...,AvgOrderValue,IsRepeatCustomer,CustomerLifetimeDays,CartAbandonmentRate,HasCoupon,DayOfWeek,IsWeekend,IsCancelled,IsReturned,IsProblemOrder
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,...,2853.10,0,0,28.6,1,Wednesday,0,0,0,0
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,...,302.70,0,0,33.3,1,Friday,0,0,0,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,...,2753.40,0,0,37.5,1,Tuesday,0,1,0,1
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,...,273.19,0,0,80.0,1,Sunday,1,0,1,1
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,...,2504.04,0,0,50.0,1,Thursday,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,...,107.04,0,0,83.3,1,Thursday,0,1,0,1
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,...,1325.06,0,0,60.0,0,Monday,0,1,0,1
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,...,873.68,0,0,0.0,1,Thursday,0,0,0,0
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,...,1050.08,0,0,0.0,1,Thursday,0,1,0,1


In [ ]:
orders_clean.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice,CalculatedTotal,PriceDiscrepancy,Year,Quarter,Unattendedorders,...,TotalSpent,AvgOrderValue,IsRepeatCustomer,CustomerLifetimeDays,CartAbandonmentRate,HasCoupon,IsWeekend,IsCancelled,IsReturned,IsProblemOrder
count,1200,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1.200000e+03,1200.000000,1200.000000,1200.000000,...,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300,1053.968300,2.475057e-15,2023.767500,2.317500,2.539167,...,1070.248117,1053.968300,0.018333,6.783333,42.074000,0.742500,0.298333,0.208333,0.205833,0.414167
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000,11.390000,-4.547474e-13,2023.000000,1.000000,0.000000,...,11.390000,11.390000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000,410.520000,0.000000e+00,2023.000000,1.000000,1.000000,...,418.037500,410.877500,0.000000,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000,823.615000,0.000000e+00,2024.000000,2.000000,3.000000,...,840.270000,825.340000,0.000000,0.000000,50.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000,1578.475000,0.000000e+00,2024.000000,3.000000,4.000000,...,1603.790000,1575.862500,0.000000,0.000000,60.000000,1.000000,1.000000,0.000000,0.000000,1.000000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000,3456.400000,4.547474e-13,2025.000000,4.000000,5.000000,...,5723.230000,3456.400000,1.000000,718.000000,83.300000,1.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,1.407557,197.177146,2.281983,819.856558,819.856558,7.923713e-14,0.750942,1.089043,1.735697,...,839.904318,817.418976,0.134210,54.438246,24.835735,0.437439,0.457717,0.406286,0.404478,0.492783
